# ISIC 2024 - Metadata-Only Deep Learning Pipeline
Notebook này chạy huấn luyện và suy diễn chỉ bằng dữ liệu bảng (metadata) của cuộc thi ISIC 2024.
Môi trường chạy thích hợp: Kaggle Notebook (có GPU hỗ trợ tăng tốc).


In [ ]:
# Setup Paths & Environment
from pathlib import Path
import sys
import os

is_kaggle = Path('/kaggle/input').exists()
if is_kaggle:
    COMP_DATA_ROOT = Path('/kaggle/input/isic-2024-challenge')
    OUTPUT_DIR = Path('/kaggle/working/outputs')
else:
    # Local config for testing
    COMP_DATA_ROOT = Path('../data')  # Adjust as needed
    OUTPUT_DIR = Path('./outputs')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('✅ Environment:', 'KAGGLE' if is_kaggle else 'LOCAL')
print('✅ Data Root:', COMP_DATA_ROOT)
print('✅ Output Dir:', OUTPUT_DIR)


In [ ]:
# Pure PyTorch TabR model implementation requires no custom library packages.
print('TabR will run entirely on standard PyTorch and Scikit-Learn.')


In [ ]:
# Imports
import gc
import json
import random
import warnings
from dataclasses import dataclass
from typing import Optional, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import OneHotEncoder, StandardScaler, QuantileTransformer

warnings.filterwarnings('ignore')

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_seed(42)


## 1. Feature Engineering & Preprocessing Pipeline


In [ ]:
# =========================================================
# FEATURE DEFINITIONS & DATA PIPELINE
# =========================================================

# --- Config Variables ---
import itertools
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Optional

BASE_NUMERIC_COLS = [
    "age_approx",
    "clin_size_long_diam_mm",
    "tbp_lv_A",
    "tbp_lv_Aext",
    "tbp_lv_B",
    "tbp_lv_Bext",
    "tbp_lv_C",
    "tbp_lv_Cext",
    "tbp_lv_H",
    "tbp_lv_Hext",
    "tbp_lv_L",
    "tbp_lv_Lext",
    "tbp_lv_areaMM2",
    "tbp_lv_area_perim_ratio",
    "tbp_lv_color_std_mean",
    "tbp_lv_deltaA",
    "tbp_lv_deltaB",
    "tbp_lv_deltaL",
    "tbp_lv_deltaLB",
    "tbp_lv_deltaLBnorm",
    "tbp_lv_eccentricity",
    "tbp_lv_minorAxisMM",
    "tbp_lv_nevi_confidence",
    "tbp_lv_norm_border",
    "tbp_lv_norm_color",
    "tbp_lv_perimeterMM",
    "tbp_lv_radial_color_std_max",
    "tbp_lv_stdL",
    "tbp_lv_stdLExt",
    "tbp_lv_symm_2axis",
    "tbp_lv_symm_2axis_angle",
    "tbp_lv_x",
    "tbp_lv_y",
    "tbp_lv_z",
]

ENGINEERED_NUMERIC_COLS = [
    "lesion_size_ratio",
    "lesion_shape_index",
    "hue_contrast",
    "luminance_contrast",
    "lesion_color_difference",
    "border_complexity",
    "color_uniformity",
    "position_distance_3d",
    "perimeter_to_area_ratio",
    # Removed: "area_to_perimeter_ratio" (inverse of perimeter_to_area_ratio)
    "lesion_visibility_score",
    "symmetry_border_consistency",
    # Removed: "consistency_symmetry_border" (duplicate of symmetry_border_consistency)
    "color_consistency",
    # Removed: "consistency_color" (duplicate of color_consistency)
    "size_age_interaction",
    "hue_color_std_interaction",
    "lesion_severity_index",
    "shape_complexity_index",
    "color_contrast_index",
    "log_lesion_area",
    "normalized_lesion_size",
    "mean_hue_difference",
    "std_dev_contrast",
    "color_shape_composite_index",
    "lesion_orientation_3d",
    # Removed: "overall_color_difference" (redundant with color_difference_total)
    "symmetry_perimeter_interaction",
    "comprehensive_lesion_index",
    "color_variance_ratio",
    "border_color_interaction",
    # Removed: "border_color_interaction_2" (redundant logic with border_color_interaction)
    "size_color_contrast_ratio",
    "age_normalized_nevi_confidence",
    # Removed: "age_normalized_nevi_confidence_2" (invalid combination of size + age)
    "color_asymmetry_index",
    "volume_approximation_3d",
    "color_range",
    "shape_color_consistency",
    "border_length_ratio",
    "age_size_symmetry_index",
    # Removed: "index_age_size_symmetry" (duplicate 3-way interaction with age_size_symmetry_index)
    "asymmetry_ratio",
    "asymmetry_area_ratio",
    "color_variation_intensity",
    "color_contrast_ratio",
    "shape_irregularity",
    "border_density",
    "size_age_ratio",
    "area_diameter_ratio",
    "position_norm_3d",
    # Removed: "position_angle_3d_xz" (redundant 3D angle)
    # Removed: "position_angle_3d_yz" (redundant 3D angle)
    "lab_chroma",
    "lab_hue",
    "texture_contrast",
    "texture_uniformity",
    "color_difference_AB",
    "color_difference_total",
    "color_change_ratio",
    "lesion_complexity_index",
    "age_adjusted_lesion_size",
    "color_uniformity_index",
    "border_sharpness_index",
    # Removed: "relative_depth_3d" (unnecessary 3D depth)
    "lesion_aspect_ratio",
    "color_contrast_variation",
    "symmetry_color_index",
    "age_adjusted_complexity",
    "relative_lesion_size",
    "color_gradient_intensity",
    "lesion_density_index",
    "color_diversity_index",
    "border_irregularity_size_ratio",
    "estimated_growth_rate",
    "color_asymmetry_ratio",
    # Removed: "sphericity_3d" (lesion is 2D on skin, not truly 3D)
    "relative_position_x",
    "relative_position_y",
    "malignancy_composite_index",
    "lesion_complexity_index_patient_variation",
    "color_diversity_index_patient_variation",
    "border_irregularity_size_ratio_patient_variation",
    "tbp_lv_areaMM2_anatomical_zscore",
    "tbp_lv_norm_border_anatomical_zscore",
    "tbp_lv_norm_color_anatomical_zscore",
    "tbp_lv_areaMM2_patient",
    "tbp_lv_areaMM2_bp",
    "patient_lof_score",
]

DEFAULT_CATEGORICAL_COLS = [
    "sex",
    "anatom_site_general",
    "tbp_tile_type",
    "tbp_lv_location",
    "tbp_lv_location_simple",
    "attribution",
]

CONTEXT_NORM_SUFFIXES = [
    "patient_norm",  # Z-score normalized by patient
    "patient_site_general_norm",  # Z-score normalized by anatomical site
]

UGLY_DUCKLING_SOURCE_COLS = list(BASE_NUMERIC_COLS) + list(ENGINEERED_NUMERIC_COLS)
UGLY_DUCKLING_SUMMARY_COLS = [
    "ud_count_location",
    "ud_max_severity_location",
    "ud_consistency_location",
]


def _deduplicate_keep_order(values: list[str]) -> list[str]:
    return list(dict.fromkeys(values))


CONTEXT_NUMERIC_COLS = [
    f"{col}_{suffix}"
    for suffix in CONTEXT_NORM_SUFFIXES
    for col in (BASE_NUMERIC_COLS + ENGINEERED_NUMERIC_COLS)
]

UGLY_DUCKLING_NUMERIC_COLS = [f"ud_loc_{col}" for col in UGLY_DUCKLING_SOURCE_COLS] + [
    f"ud_percentile_{col}" for col in UGLY_DUCKLING_SOURCE_COLS
]

CATEGORICAL_COUNT_COLS = [f"{col}_count" for col in DEFAULT_CATEGORICAL_COLS]

NUM_CAT_INTERACTION_COLS = [
    f"{num_col}_{cat_col}"
    for num_col, cat_col in itertools.product(
        BASE_NUMERIC_COLS, DEFAULT_CATEGORICAL_COLS
    )
]

DEFAULT_NUMERIC_COLS = _deduplicate_keep_order(
    list(BASE_NUMERIC_COLS)
    + list(ENGINEERED_NUMERIC_COLS)
    + CONTEXT_NUMERIC_COLS
    + UGLY_DUCKLING_NUMERIC_COLS
    + UGLY_DUCKLING_SUMMARY_COLS
    + ["count_per_patient"]
    + CATEGORICAL_COUNT_COLS
    + NUM_CAT_INTERACTION_COLS
)

# --- Data Preparation ---
"""Data preparation utilities for numeric column preprocessing."""


import numpy as np
import pandas as pd



def _prepare_base_numeric(df: pd.DataFrame, err: float) -> pd.DataFrame:
    """Prepare numeric columns: convert types, handle missing values, and fix infinities.

    Args:
        df: Input DataFrame with potentially mixed types and missing values.
        err: Small epsilon value for numerical stability.

    Returns:
        DataFrame with numeric columns properly typed and cleaned (NaN-filled with medians,
        infinities replaced, converted to float32).
    """
    out = df.copy()

    if "age_approx" in out.columns:
        out["age_approx"] = (
            out["age_approx"].replace("NA", np.nan).pipe(pd.to_numeric, errors="coerce")
        )

    base_cols = [c for c in BASE_NUMERIC_COLS if c in out.columns]
    for col in base_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    if base_cols:
        out[base_cols] = out[base_cols].replace([np.inf, -np.inf], np.nan)
        medians = out[base_cols].median(numeric_only=True)
        out[base_cols] = out[base_cols].fillna(medians)
        out[base_cols] = out[base_cols].astype(np.float32, copy=False)

    return out


def _check_required_columns(df: pd.DataFrame, group_col: str) -> None:
    """Validate that all required columns are present in DataFrame.

    Args:
        df: Input DataFrame to validate.
        group_col: Name of the grouping column (e.g., 'patient_id').

    Raises:
        KeyError: If any required columns are missing.
    """
    required = set(BASE_NUMERIC_COLS)
    required.add(group_col)
    missing = sorted(col for col in required if col not in df.columns)
    if missing:
        raise KeyError(
            "Missing required columns for feature engineering: " + ", ".join(missing)
        )

# --- Null Importance Feature Selection ---
"""Feature selection utilities using correlation and null importance methods."""


import numpy as np
import pandas as pd


def _absolute_feature_target_corr(matrix: np.ndarray, target: np.ndarray) -> np.ndarray:
    """Compute absolute Pearson correlation between features and target.

    Args:
        matrix: Feature matrix of shape (n_samples, n_features), dtype float32.
        target: Target vector of shape (n_samples,), dtype float32.

    Returns:
        Array of absolute correlation coefficients for each feature, shape (n_features,).
    """
    centered_x = matrix - matrix.mean(axis=0, keepdims=True)
    centered_y = target - target.mean()

    numerator = np.abs(np.sum(centered_x * centered_y[:, None], axis=0))
    denominator = np.sqrt(
        np.sum(centered_x**2, axis=0) * np.sum(centered_y**2, axis=0) + 1e-8
    )
    corr = numerator / denominator
    corr = np.nan_to_num(corr, nan=0.0, posinf=0.0, neginf=0.0)
    return corr.astype(np.float32)


def _select_numeric_by_null_importance(
    numeric_frame: pd.DataFrame,
    target: pd.Series,
    repeats: int,
    quantile: float,
    min_kept: int,
    random_state: int,
) -> list[str]:
    """Filter numeric features using null importance permutation method.

    Compares feature-target correlation under actual target vs shuffled target.
    Keeps features where the margin (real_score - null_threshold) > 0, ensuring
    at least min_kept features are retained.

    Args:
        numeric_frame: DataFrame of numeric features.
        target: Target series for correlation computation.
        repeats: Number of random shuffles for null distribution.
        quantile: Quantile of null scores to use as threshold (0.5-0.999).
        min_kept: Minimum number of features to retain.
        random_state: Random seed for reproducibility.

    Returns:
        List of selected feature column names.
    """
    if numeric_frame.empty:
        return []

    matrix = numeric_frame.to_numpy(dtype=np.float32, copy=True)
    matrix = np.nan_to_num(matrix, nan=0.0, posinf=0.0, neginf=0.0)

    y = pd.to_numeric(target, errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
    if len(y) < 16:
        return list(numeric_frame.columns)

    real_score = _absolute_feature_target_corr(matrix=matrix, target=y)

    n_repeats = max(1, int(repeats))
    clipped_quantile = float(np.clip(quantile, 0.5, 0.999))
    rng = np.random.default_rng(random_state)
    null_scores = np.zeros((n_repeats, matrix.shape[1]), dtype=np.float32)

    for i in range(n_repeats):
        shuffled = rng.permutation(y)
        null_scores[i] = _absolute_feature_target_corr(matrix=matrix, target=shuffled)

    null_threshold = np.quantile(null_scores, clipped_quantile, axis=0)
    margin = real_score - null_threshold
    keep_mask = margin > 0.0

    min_kept = max(1, min(int(min_kept), matrix.shape[1]))
    if int(keep_mask.sum()) < min_kept:
        keep_mask = np.zeros_like(keep_mask, dtype=bool)
        top_idx = np.argsort(margin)[-min_kept:]
        keep_mask[top_idx] = True

    return [
        col
        for col, keep in zip(numeric_frame.columns, keep_mask.tolist())
        if bool(keep)
    ]

# --- Feature Engineering ---
"""Feature engineering for medical imaging tabular data (skin lesion analysis)."""


import gc
import numpy as np
import pandas as pd
from sklearn.neighbors import LocalOutlierFactor




def _add_ugly_duckling_features(
    out: pd.DataFrame,
    generated_features: dict[str, pd.Series],
    group_col: str,
    source_cols: list[str],
    err: float,
) -> None:
    """Add 'ugly duckling' features for anomaly detection within patient cohorts.

    The 'ugly duckling' concept: lesions that look unusual compared to a patient's
    other lesions may indicate malignancy. Creates z-scores by location and adds
    aggregate anomaly indicators.

    Args:
        out: DataFrame with all data.
        generated_features: Dictionary to accumulate new features.
        group_col: Grouping column (e.g., 'patient_id').
        source_cols: Base columns to compute anomaly scores.
        err: Epsilon for numerical stability.
    """
    loc_col = None
    for candidate in [
        "tbp_lv_location",
        "tbp_lv_location_simple",
        "anatom_site_general",
    ]:
        if candidate in out.columns:
            loc_col = candidate
            break
    if loc_col is None:
        return

    patient_grouped = out.groupby(group_col, observed=False, sort=False)
    loc_grouped = out.groupby([group_col, loc_col], observed=False, sort=False)

    ud_loc_cols: list[str] = []
    for col in source_cols:
        loc_mean = loc_grouped[col].transform("mean").astype(np.float32, copy=False)
        loc_std = loc_grouped[col].transform("std").astype(np.float32, copy=False)

        ud_loc_col = f"ud_loc_{col}"
        generated_features[ud_loc_col] = ((out[col] - loc_mean) / (loc_std + err)).astype(np.float32, copy=False)
        ud_loc_cols.append(ud_loc_col)

        generated_features[f"ud_percentile_{col}"] = patient_grouped[col].rank(pct=True).astype(np.float32, copy=False)

    if not ud_loc_cols:
        return

    ud_abs = pd.DataFrame(
        {col: generated_features[col].abs() for col in ud_loc_cols},
        index=out.index,
    )
    generated_features["ud_count_location"] = (ud_abs > 2.0).sum(axis=1)
    generated_features["ud_max_severity_location"] = ud_abs.max(axis=1)
    generated_features["ud_consistency_location"] = ud_abs.std(axis=1)


def _add_contextual_group_norm_features(
    out: pd.DataFrame,
    generated_features: dict[str, pd.Series],
    group_col: str,
    source_cols: list[str],
    err: float,
) -> None:
    """Add features normalized by anatomical context (location/site).

    Creates z-score normalized features grouped by patient AND anatomical location,
    capturing how features deviate within anatomical regions.

    Args:
        out: DataFrame with all data.
        generated_features: Dictionary to accumulate new features.
        group_col: Grouping column (e.g., 'patient_id').
        source_cols: Base columns to normalize.
        err: Epsilon for numerical stability.
    """
    context_groups = [
        ("patient_site_general_norm", "anatom_site_general"),
    ]

    for suffix, context_col in context_groups:
        if context_col not in out.columns:
            continue

        # Pre-compute group statistics for ALL columns at once (cache-friendly)
        grouped = out.groupby([group_col, context_col], observed=False, sort=False)

        # Transform using cached statistics (vectorized)
        for col in source_cols:
            group_mean = grouped[col].transform("mean").astype(np.float32, copy=False)
            group_std = grouped[col].transform("std").astype(np.float32, copy=False)
            generated_features[f"{col}_{suffix}"] = ((out[col] - group_mean) / (
                group_std + err
            )).astype(np.float32, copy=False)


def _add_patient_lof_features(
    out: pd.DataFrame,
    generated_features: dict[str, pd.Series],
    group_col: str,
) -> None:
    """Add Local Outlier Factor (LOF) score for each lesion per patient cohort.

    The score indicates how much a lesion's visual/shape properties deviate from
    the patient's typical lesions (acting as a multi-dimensional ugly duckling indicator).
    """
    lof_features = [
        "clin_size_long_diam_mm",
        "tbp_lv_areaMM2",
        "tbp_lv_color_std_mean",
        "tbp_lv_symm_2axis",
        "tbp_lv_norm_border",
        "tbp_lv_norm_color",
    ]
    # Filter only columns that exist in the dataframe
    lof_features = [col for col in lof_features if col in out.columns]
    if not lof_features:
        generated_features["patient_lof_score"] = pd.Series(1.0, index=out.index)
        return

    # Fill NaNs/Infs for security
    data_subset = out[[group_col] + lof_features].copy()
    for col in lof_features:
        data_subset[col] = data_subset[col].replace([np.inf, -np.inf], np.nan)
        data_subset[col] = data_subset[col].fillna(data_subset[col].median())

    scores = np.ones(len(out), dtype=np.float32)

    # Group by patient
    grouped = data_subset.groupby(group_col, observed=False, sort=False)
    for _, group in grouped:
        n_samples = len(group)
        if n_samples >= 3:
            n_neighbors = min(20, n_samples - 1)
            if n_neighbors >= 2:
                try:
                    lof = LocalOutlierFactor(n_neighbors=n_neighbors, novelty=False)
                    lof.fit(group[lof_features])
                    scores[group.index] = -lof.negative_outlier_factor_
                except Exception:
                    pass

    generated_features["patient_lof_score"] = pd.Series(scores, index=out.index)


def _add_engineered_columns(
    df: pd.DataFrame,
    group_col: str,
    categorical_cols: list[str],
    err: float,
    use_contextual_group_norm_features: bool,
    use_ugly_duckling_features: bool,
    use_anatomical_zscore_features: bool,
    use_area_aggregation_features: bool,
) -> pd.DataFrame:
    """Main orchestrator for all feature engineering transformations.

    Generates 80+ derived features from base medical measurements including:
    - Geometric ratios and indices (lesion shape, border properties)
    - Color features (LAB space metrics, contrast, uniformity)
    - 3D spatial features (position, volume approximation)
    - Interaction features (size × age, color × symmetry)
    - Patient/cohort normalized features
    - Contextual (anatomical) normalized features
    - Anomaly detection features (ugly duckling)
    - Patient-specific aggregate features

    Args:
        df: Input DataFrame with base numeric and categorical columns.
        group_col: Column name for patient/group identifier.
        categorical_cols: Categorical column names for encoding.
        err: Epsilon for numerical stability (default: 1e-5).
        use_contextual_group_norm_features: Enable location-based normalization.
        use_ugly_duckling_features: Enable anomaly detection features.
        use_anatomical_zscore_features: Enable anatomical z-score features.
        use_area_aggregation_features: Enable area aggregation by patient/anatomical site.

    Returns:
        DataFrame with original + all engineered features, numeric columns as float32.
    """
    out = _prepare_base_numeric(df=df, err=err)
    _check_required_columns(df=out, group_col=group_col)

    out["lesion_size_ratio"] = out["tbp_lv_minorAxisMM"] / (
        out["clin_size_long_diam_mm"] + err
    )
    out["lesion_shape_index"] = out["tbp_lv_areaMM2"] / (
        np.square(out["tbp_lv_perimeterMM"]) + err
    )
    out["hue_contrast"] = np.abs(out["tbp_lv_H"] - out["tbp_lv_Hext"])
    out["luminance_contrast"] = np.abs(out["tbp_lv_L"] - out["tbp_lv_Lext"])
    out["lesion_color_difference"] = np.sqrt(
        np.square(out["tbp_lv_deltaA"])
        + np.square(out["tbp_lv_deltaB"])
        + np.square(out["tbp_lv_deltaL"])
    )
    out["border_complexity"] = out["tbp_lv_norm_border"] + out["tbp_lv_symm_2axis"]
    out["color_uniformity"] = out["tbp_lv_color_std_mean"] / (
        out["tbp_lv_radial_color_std_max"] + err
    )
    out["position_distance_3d"] = np.sqrt(
        np.square(out["tbp_lv_x"])
        + np.square(out["tbp_lv_y"])
        + np.square(out["tbp_lv_z"])
    )
    out["perimeter_to_area_ratio"] = out["tbp_lv_perimeterMM"] / (
        out["tbp_lv_areaMM2"] + err
    )
    out["lesion_visibility_score"] = (
        out["tbp_lv_deltaLBnorm"] + out["tbp_lv_norm_color"]
    )
    out["symmetry_border_consistency"] = (
        out["tbp_lv_symm_2axis"] * out["tbp_lv_norm_border"]
    )

    out["color_consistency"] = out["tbp_lv_stdL"] / (out["tbp_lv_Lext"] + err)
    out["size_age_interaction"] = out["clin_size_long_diam_mm"] * out["age_approx"]
    out["hue_color_std_interaction"] = out["tbp_lv_H"] * out["tbp_lv_color_std_mean"]
    out["lesion_severity_index"] = (
        out["tbp_lv_norm_border"]
        + out["tbp_lv_norm_color"]
        + out["tbp_lv_eccentricity"]
    ) / 3.0
    out["shape_complexity_index"] = out["border_complexity"] + out["lesion_shape_index"]
    out["color_contrast_index"] = (
        out["tbp_lv_deltaA"]
        + out["tbp_lv_deltaB"]
        + out["tbp_lv_deltaL"]
        + out["tbp_lv_deltaLBnorm"]
    )

    out["log_lesion_area"] = np.log(out["tbp_lv_areaMM2"] + 1.0)
    out["normalized_lesion_size"] = out["clin_size_long_diam_mm"] / (
        out["age_approx"] + err
    )
    out["mean_hue_difference"] = (out["tbp_lv_H"] + out["tbp_lv_Hext"]) / 2.0
    out["std_dev_contrast"] = np.sqrt(
        (
            np.square(out["tbp_lv_deltaA"])
            + np.square(out["tbp_lv_deltaB"])
            + np.square(out["tbp_lv_deltaL"])
        )
        / 3.0
    )
    out["color_shape_composite_index"] = (
        out["tbp_lv_color_std_mean"]
        + out["tbp_lv_area_perim_ratio"]
        + out["tbp_lv_symm_2axis"]
    ) / 3.0
    out["lesion_orientation_3d"] = np.arctan2(out["tbp_lv_y"], out["tbp_lv_x"])

    out["symmetry_perimeter_interaction"] = (
        out["tbp_lv_symm_2axis"] * out["tbp_lv_perimeterMM"]
    )
    out["comprehensive_lesion_index"] = (
        out["tbp_lv_area_perim_ratio"]
        + out["tbp_lv_eccentricity"]
        + out["tbp_lv_norm_color"]
        + out["tbp_lv_symm_2axis"]
    ) / 4.0
    out["color_variance_ratio"] = out["tbp_lv_color_std_mean"] / (
        out["tbp_lv_stdLExt"] + err
    )
    out["border_color_interaction"] = (
        out["tbp_lv_norm_border"] * out["tbp_lv_norm_color"]
    )
    out["size_color_contrast_ratio"] = out["clin_size_long_diam_mm"] / (
        out["tbp_lv_deltaLBnorm"] + err
    )
    out["age_normalized_nevi_confidence"] = out["tbp_lv_nevi_confidence"] / (
        out["age_approx"] + err
    )
    out["color_asymmetry_index"] = (
        out["tbp_lv_radial_color_std_max"] * out["tbp_lv_symm_2axis"]
    )

    out["volume_approximation_3d"] = out["tbp_lv_areaMM2"] * np.sqrt(
        np.square(out["tbp_lv_x"])
        + np.square(out["tbp_lv_y"])
        + np.square(out["tbp_lv_z"])
    )
    out["color_range"] = (
        np.abs(out["tbp_lv_L"] - out["tbp_lv_Lext"])
        + np.abs(out["tbp_lv_A"] - out["tbp_lv_Aext"])
        + np.abs(out["tbp_lv_B"] - out["tbp_lv_Bext"])
    )
    out["shape_color_consistency"] = (
        out["tbp_lv_eccentricity"] * out["tbp_lv_color_std_mean"]
    )
    out["border_length_ratio"] = out["tbp_lv_perimeterMM"] / (
        2.0 * np.pi * np.sqrt(out["tbp_lv_areaMM2"] / np.pi + err)
    )
    out["age_size_symmetry_index"] = (
        out["age_approx"] * out["clin_size_long_diam_mm"] * out["tbp_lv_symm_2axis"]
    )

    additional_features = {
        "asymmetry_ratio": out["tbp_lv_symm_2axis"]
        / (np.abs(out["tbp_lv_symm_2axis_angle"]) + err),
        "asymmetry_area_ratio": (out["tbp_lv_symm_2axis"] * out["tbp_lv_areaMM2"])
        / (out["tbp_lv_perimeterMM"] + err),
        "color_variation_intensity": out["tbp_lv_color_std_mean"]
        * out["tbp_lv_radial_color_std_max"],
        "color_contrast_ratio": out["lesion_color_difference"]
        / (out["tbp_lv_color_std_mean"] + err),
        "shape_irregularity": out["tbp_lv_perimeterMM"]
        / (2.0 * np.sqrt(np.pi * out["tbp_lv_areaMM2"] + err)),
        "border_density": out["tbp_lv_norm_border"] / (out["tbp_lv_areaMM2"] + err),
        "size_age_ratio": out["tbp_lv_areaMM2"] / (out["age_approx"] + err),
        "area_diameter_ratio": out["tbp_lv_areaMM2"]
        / (out["clin_size_long_diam_mm"] + err),
        "position_norm_3d": out["position_distance_3d"] / (out["tbp_lv_areaMM2"] + err),
        "lab_chroma": np.sqrt(np.square(out["tbp_lv_A"]) + np.square(out["tbp_lv_B"])),
        "lab_hue": np.arctan2(out["tbp_lv_B"], out["tbp_lv_A"]),
        "texture_contrast": out["tbp_lv_stdL"] / (out["tbp_lv_stdLExt"] + err),
        "texture_uniformity": 1.0 / (out["tbp_lv_color_std_mean"] + err),
        "color_difference_AB": np.abs(out["tbp_lv_deltaA"])
        + np.abs(out["tbp_lv_deltaB"]),
        "color_difference_total": np.abs(out["tbp_lv_deltaA"])
        + np.abs(out["tbp_lv_deltaB"])
        + np.abs(out["tbp_lv_deltaL"]),
    }
    additional_features["color_change_ratio"] = additional_features[
        "color_difference_total"
    ] / (np.abs(out["tbp_lv_deltaLBnorm"]) + err)
    additional_features["lesion_complexity_index"] = (
        out["shape_complexity_index"] * out["color_variance_ratio"]
    )
    additional_features["age_adjusted_lesion_size"] = out["log_lesion_area"] / (
        out["age_approx"] + err
    )
    additional_features["color_uniformity_index"] = 1.0 / (
        out["color_uniformity"] + err
    )
    additional_features["border_sharpness_index"] = out["tbp_lv_norm_border"] / (
        out["tbp_lv_color_std_mean"] + err
    )
    additional_features["lesion_aspect_ratio"] = out["clin_size_long_diam_mm"] / (
        out["tbp_lv_minorAxisMM"] + err
    )
    additional_features["color_contrast_variation"] = (
        out["color_contrast_index"] * out["tbp_lv_color_std_mean"]
    )
    additional_features["symmetry_color_index"] = (
        out["tbp_lv_symm_2axis"] * out["tbp_lv_norm_color"]
    )
    additional_features["age_adjusted_complexity"] = additional_features[
        "lesion_complexity_index"
    ] / (out["age_approx"] + err)
    additional_features["color_gradient_intensity"] = additional_features[
        "color_difference_total"
    ] / (out["tbp_lv_perimeterMM"] + err)
    additional_features["lesion_density_index"] = out["lesion_color_difference"] / (
        out["tbp_lv_areaMM2"] + err
    )
    additional_features["color_diversity_index"] = (
        out["color_range"] * out["color_variance_ratio"]
    )
    additional_features["border_irregularity_size_ratio"] = out["border_complexity"] / (
        out["clin_size_long_diam_mm"] + err
    )
    additional_features["estimated_growth_rate"] = out["tbp_lv_areaMM2"] / (
        out["age_approx"] + err
    )
    additional_features["color_asymmetry_ratio"] = out[
        "tbp_lv_radial_color_std_max"
    ] / (out["tbp_lv_symm_2axis"] + err)
    additional_features["relative_position_x"] = out["tbp_lv_x"] / (
        out["position_distance_3d"] + err
    )
    additional_features["relative_position_y"] = out["tbp_lv_y"] / (
        out["position_distance_3d"] + err
    )
    additional_features["malignancy_composite_index"] = (
        out["lesion_severity_index"]
        + additional_features["lesion_complexity_index"]
        + out["lesion_visibility_score"]
    ) / 3.0
    out = pd.concat(
        [out, pd.DataFrame(additional_features, index=out.index)],
        axis=1,
        copy=False,
    )

    if group_col in out.columns:
        out[group_col] = (
            out[group_col].astype("object").fillna("UNK").astype(str).astype("category")
        )

    for col in categorical_cols:
        if col in out.columns:
            out[col] = (
                out[col].astype("object").fillna("UNK").astype(str).astype("category")
            )

    patient_grouped = out.groupby(group_col, observed=False, sort=False)
    patient_variation_features = {
        "relative_lesion_size": out["clin_size_long_diam_mm"]
        / (patient_grouped["clin_size_long_diam_mm"].transform("mean") + err),
        "lesion_complexity_index_patient_variation": out["lesion_complexity_index"]
        - patient_grouped["lesion_complexity_index"].transform("mean"),
        "color_diversity_index_patient_variation": out["color_diversity_index"]
        - patient_grouped["color_diversity_index"].transform("mean"),
        "border_irregularity_size_ratio_patient_variation": out[
            "border_irregularity_size_ratio"
        ]
        - patient_grouped["border_irregularity_size_ratio"].transform("mean"),
    }
    out = pd.concat(
        [out, pd.DataFrame(patient_variation_features, index=out.index)],
        axis=1,
        copy=False,
    )
    # Cast all existing numeric columns to float32 early to save memory before heavy group normalization
    num_cols = out.select_dtypes(include=[np.number]).columns
    out[num_cols] = out[num_cols].astype(np.float32, copy=False)
    gc.collect()

    patient_grouped = out.groupby(group_col, observed=False, sort=False)

    generated_features: dict[str, pd.Series] = {}
    source_cols = [
        col
        for col in (BASE_NUMERIC_COLS + ENGINEERED_NUMERIC_COLS)
        if col in out.columns
    ]

    for col in source_cols:
        group_mean = patient_grouped[col].transform("mean").astype(np.float32, copy=False)
        group_std = patient_grouped[col].transform("std").astype(np.float32, copy=False)
        generated_features[f"{col}_patient_norm"] = ((out[col] - group_mean) / (
            group_std + err
        )).astype(np.float32, copy=False)

    if use_contextual_group_norm_features:
        _add_contextual_group_norm_features(
            out=out,
            generated_features=generated_features,
            group_col=group_col,
            source_cols=source_cols,
            err=err,
        )

    if use_area_aggregation_features and "tbp_lv_areaMM2" in out.columns:
        generated_features["tbp_lv_areaMM2_patient"] = patient_grouped[
            "tbp_lv_areaMM2"
        ].transform("sum")
        if "anatom_site_general" in out.columns:
            generated_features["tbp_lv_areaMM2_bp"] = out.groupby(
                [group_col, "anatom_site_general"], observed=False, sort=False
            )["tbp_lv_areaMM2"].transform("sum")

    if use_anatomical_zscore_features and "anatom_site_general" in out.columns:
        site_grouped = out.groupby("anatom_site_general", observed=False, sort=False)
        for col in ["tbp_lv_areaMM2", "tbp_lv_norm_border", "tbp_lv_norm_color"]:
            if col not in out.columns:
                continue
            site_mean = site_grouped[col].transform("mean")
            site_std = site_grouped[col].transform("std")
            generated_features[f"{col}_anatomical_zscore"] = (out[col] - site_mean) / (
                site_std + err
            )

    if use_ugly_duckling_features:
        _add_ugly_duckling_features(
            out=out,
            generated_features=generated_features,
            group_col=group_col,
            source_cols=source_cols,
            err=err,
        )
        _add_patient_lof_features(
            out=out,
            generated_features=generated_features,
            group_col=group_col,
        )

    for cat_col in categorical_cols:
        if cat_col not in out.columns:
            continue
        grouped = out.groupby([group_col, cat_col], observed=False, sort=False)
        for num_col in BASE_NUMERIC_COLS:
            if num_col not in out.columns:
                continue
            group_mean = grouped[num_col].transform("mean").astype(np.float32, copy=False)
            group_std = grouped[num_col].transform("std").astype(np.float32, copy=False)
            generated_features[f"{num_col}_{cat_col}"] = ((out[num_col] - group_mean) / (
                group_std + err
            )).astype(np.float32, copy=False)

    for col in categorical_cols:
        if col not in out.columns:
            continue
        generated_features[f"{col}_count"] = out.groupby(
            [group_col, col], observed=False, sort=False
        )[col].transform("count")

    generated_features["count_per_patient"] = patient_grouped[group_col].transform(
        "count"
    )

    if generated_features:
        # Explicitly cast all generated features to float32 before concatenating
        for k in list(generated_features.keys()):
            if np.issubdtype(generated_features[k].dtype, np.number):
                generated_features[k] = generated_features[k].astype(np.float32, copy=False)
        
        out = pd.concat(
            [out, pd.DataFrame(generated_features, index=out.index)],
            axis=1,
            copy=False,
        )
        del generated_features
        gc.collect()

    numeric_cols = out.select_dtypes(include=[np.number]).columns
    out[numeric_cols] = out[numeric_cols].replace([np.inf, -np.inf], np.nan)
    out[numeric_cols] = out[numeric_cols].fillna(out[numeric_cols].median())
    out[numeric_cols] = out[numeric_cols].astype(np.float32, copy=False)

    return out


def add_feature_engineering(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    group_col: str,
    categorical_cols: list[str],
    err: float = 1e-5,
    use_contextual_group_norm_features: bool = True,
    use_ugly_duckling_features: bool = True,
    use_anatomical_zscore_features: bool = True,
    use_area_aggregation_features: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply complete feature engineering pipeline to train and test sets.

    Wraps _add_engineered_columns for both datasets with consistent parameters,
    ensuring train and test receive identical feature transformations.

    Args:
        train_df: Training DataFrame.
        test_df: Test DataFrame.
        group_col: Column name for grouping (e.g., 'patient_id').
        categorical_cols: List of categorical column names.
        err: Epsilon for numerical stability (default: 1e-5).
        use_contextual_group_norm_features: Enable location-based normalization (default: True).
        use_ugly_duckling_features: Enable anomaly detection (default: True).
        use_anatomical_zscore_features: Enable anatomical z-score (default: True).
        use_area_aggregation_features: Enable area aggregation (default: True).

    Returns:
        Tuple of (train_enhanced, test_enhanced) DataFrames with all features.
    """
    train_enhanced = _add_engineered_columns(
        df=train_df,
        group_col=group_col,
        categorical_cols=categorical_cols,
        err=err,
        use_contextual_group_norm_features=use_contextual_group_norm_features,
        use_ugly_duckling_features=use_ugly_duckling_features,
        use_anatomical_zscore_features=use_anatomical_zscore_features,
        use_area_aggregation_features=use_area_aggregation_features,
    )
    test_enhanced = _add_engineered_columns(
        df=test_df,
        group_col=group_col,
        categorical_cols=categorical_cols,
        err=err,
        use_contextual_group_norm_features=use_contextual_group_norm_features,
        use_ugly_duckling_features=use_ugly_duckling_features,
        use_anatomical_zscore_features=use_anatomical_zscore_features,
        use_area_aggregation_features=use_area_aggregation_features,
    )
    return train_enhanced, test_enhanced

# --- Tabular Preprocessor ---
"""Preprocessing and encoding for tabular data using scalers and encoders."""


from dataclasses import dataclass
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, QuantileTransformer, StandardScaler



def _build_one_hot_encoder() -> OneHotEncoder:
    """Initialize OneHotEncoder with float32 output for compatibility.

    Returns:
        OneHotEncoder configured for handling unknown categories and float32 dtype.
    """
    try:
        return OneHotEncoder(
            handle_unknown="ignore", sparse_output=False, dtype=np.float32
        )
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False, dtype=np.float32)


def select_available_columns(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    numeric_candidates: Iterable[str],
    categorical_candidates: Iterable[str],
) -> tuple[list[str], list[str]]:
    """Select columns that exist in both train and test sets.

    Filters candidate columns to only those present in both datasets to ensure
    consistent preprocessing and feature availability.

    Args:
        train_df: Training DataFrame.
        test_df: Test DataFrame.
        numeric_candidates: Candidate numeric column names.
        categorical_candidates: Candidate categorical column names.

    Returns:
        Tuple of (numeric_cols, categorical_cols) filtered to common columns.
    """
    train_cols = set(train_df.columns)
    test_cols = set(test_df.columns)

    numeric_cols = [
        col for col in numeric_candidates if col in train_cols and col in test_cols
    ]
    categorical_cols = [
        col for col in categorical_candidates if col in train_cols and col in test_cols
    ]
    return numeric_cols, categorical_cols


@dataclass
class FoldTabularPreprocessor:
    """Preprocessing pipeline for tabular data: scaling, encoding, optional quantile transform.

    Attributes:
        numeric_cols: List of numeric column names to scale.
        categorical_cols: List of categorical column names to one-hot encode.
        use_quantile_transform: Whether to apply quantile transformer (default: False).
        quantile_noise_std: Standard deviation of Gaussian noise added before quantile transform.
        quantile_random_state: Random seed for quantile transformer.
        use_null_importance_filter: Whether to filter features by null importance (default: False).
        null_importance_repeats: Number of shuffle repeats for null importance (default: 8).
        null_importance_quantile: Quantile threshold for null importance (default: 0.95).
        null_importance_min_kept: Minimum features to keep after filtering (default: 64).
        target_col: Name of target column (required if use_null_importance_filter=True).
    """

    numeric_cols: list[str]
    categorical_cols: list[str]
    use_quantile_transform: bool = False
    quantile_noise_std: float = 1e-5
    quantile_random_state: int = 42
    use_null_importance_filter: bool = False
    null_importance_repeats: int = 8
    null_importance_quantile: float = 0.95
    null_importance_min_kept: int = 64
    target_col: str = "target"

    def __post_init__(self) -> None:
        """Initialize scalers, encoders, and preprocessing state."""
        self.numeric_cols = list(dict.fromkeys(self.numeric_cols))
        self.active_numeric_cols = list(self.numeric_cols)
        self.numeric_medians: dict[str, float] = {}
        self.scaler = StandardScaler()
        self.encoder = _build_one_hot_encoder()
        self.quantile_transformer: QuantileTransformer | None = None
        self.is_fitted = False

    def _transform_base(self, df: pd.DataFrame) -> np.ndarray:
        """Transform numeric and categorical columns into concatenated feature matrix.

        Args:
            df: Input DataFrame.

        Returns:
            Feature matrix of shape (n_samples, n_scaled + n_encoded_features).

        Raises:
            ValueError: If no columns are available for preprocessing.
        """
        chunks: list[np.ndarray] = []

        if self.active_numeric_cols:
            numeric_frame = df[self.active_numeric_cols].copy()
            numeric_frame = numeric_frame.apply(pd.to_numeric, errors="coerce")
            numeric_filled = numeric_frame.fillna(self.numeric_medians)
            numeric_scaled = self.scaler.transform(numeric_filled.values).astype(
                np.float32
            )
            chunks.append(numeric_scaled)

        if self.categorical_cols:
            cat_frame = df[self.categorical_cols].copy()
            cat_frame = cat_frame.astype("object").fillna("UNK").astype(str)
            cat_encoded = self.encoder.transform(cat_frame).astype(np.float32)
            chunks.append(cat_encoded)

        if not chunks:
            raise ValueError("No tabular columns are available after preprocessing.")

        if len(chunks) == 1:
            return chunks[0]
        return np.concatenate(chunks, axis=1)

    def fit(self, df: pd.DataFrame) -> None:
        """Fit scalers, encoders, and optionally apply null importance filtering.

        Args:
            df: Training DataFrame. Must contain numeric, categorical, and optionally target columns.
        """
        if self.active_numeric_cols:
            numeric_frame = df[self.active_numeric_cols].copy()
            numeric_frame = numeric_frame.apply(pd.to_numeric, errors="coerce")

            if self.use_null_importance_filter and self.target_col in df.columns:
                selected_numeric = _select_numeric_by_null_importance(
                    numeric_frame=numeric_frame,
                    target=df[self.target_col],
                    repeats=self.null_importance_repeats,
                    quantile=self.null_importance_quantile,
                    min_kept=self.null_importance_min_kept,
                    random_state=self.quantile_random_state,
                )
                if selected_numeric and len(selected_numeric) < len(
                    self.active_numeric_cols
                ):
                    self.active_numeric_cols = selected_numeric
                    numeric_frame = numeric_frame[self.active_numeric_cols]
                    print(
                        "Null-importance filter kept "
                        f"{len(self.active_numeric_cols)}/{len(self.numeric_cols)} "
                        "numeric features."
                    )

            self.numeric_medians = {
                col: float(numeric_frame[col].median())
                for col in self.active_numeric_cols
            }
            numeric_filled = numeric_frame.fillna(self.numeric_medians)
            self.scaler.fit(numeric_filled.values)

        if self.categorical_cols:
            cat_frame = df[self.categorical_cols].copy()
            cat_frame = cat_frame.astype("object").fillna("UNK").astype(str)
            self.encoder.fit(cat_frame)

        if self.use_quantile_transform:
            base_matrix = self._transform_base(df)

            if len(base_matrix) >= 2:
                n_quantiles = max(min(len(base_matrix) // 30, 1000), 10)
                n_quantiles = min(n_quantiles, len(base_matrix))

                noise = (
                    np.random.default_rng(self.quantile_random_state)
                    .normal(
                        loc=0.0,
                        scale=self.quantile_noise_std,
                        size=base_matrix.shape,
                    )
                    .astype(np.float32)
                )

                self.quantile_transformer = QuantileTransformer(
                    n_quantiles=n_quantiles,
                    output_distribution="normal",
                    subsample=10**9,
                    random_state=self.quantile_random_state,
                )
                self.quantile_transformer.fit(base_matrix + noise)
            else:
                self.quantile_transformer = None

        self.is_fitted = True

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        """Transform DataFrame using fitted preprocessor.

        Args:
            df: DataFrame to transform. Must contain same columns as training data.

        Returns:
            Preprocessed feature matrix of shape (n_samples, n_features).

        Raises:
            RuntimeError: If fit() has not been called yet.
        """
        if not self.is_fitted:
            raise RuntimeError(
                "FoldTabularPreprocessor must be fitted before transform."
            )

        base_matrix = self._transform_base(df)
        if self.use_quantile_transform and self.quantile_transformer is not None:
            return self.quantile_transformer.transform(base_matrix).astype(np.float32)
        return base_matrix


## 2. Shared Utilities (Evaluation & Samplers)


In [ ]:
def partial_auc_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Official competition metric: partial AUC at TPR >= 0.80."""
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    min_tpr = 0.80
    max_fpr = abs(1.0 - min_tpr)

    v_gt = np.abs(y_true - 1.0)
    v_pred = 1.0 - y_pred

    try:
        partial_auc_scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    except ValueError:
        return float("nan")

    partial_auc = 0.5 * max_fpr**2 + (
        (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (partial_auc_scaled - 0.5)
    )
    return float(partial_auc)

def make_balanced_sampler(
    targets: np.ndarray,
    negative_ratio: int,
    seed: int,
    hard_negative_weights: np.ndarray | None = None,
) -> WeightedRandomSampler | None:
    labels = np.asarray(targets, dtype=np.int64)
    pos_mask = labels == 1
    neg_mask = labels == 0

    n_pos = int(pos_mask.sum())
    n_neg = int(neg_mask.sum())
    if n_pos == 0 or n_neg == 0:
        return None

    weights = np.zeros(len(labels), dtype=np.float64)
    weights[pos_mask] = 1.0
    target_neg_sum = n_pos * max(1, negative_ratio)

    if hard_negative_weights is not None:
        neg_w = np.asarray(hard_negative_weights, dtype=np.float64)[neg_mask]
        if neg_w.sum() > 0:
            neg_w = (neg_w / neg_w.sum()) * target_neg_sum
        else:
            neg_w = target_neg_sum / max(1, n_neg)
        weights[neg_mask] = neg_w
    else:
        weights[neg_mask] = target_neg_sum / max(1, n_neg)

    num_samples = int(n_pos * (1 + max(1, negative_ratio)))
    generator = torch.Generator()
    generator.manual_seed(seed)

    return WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=num_samples,
        replacement=True,
        generator=generator,
    )

class TabularDataset(Dataset):
    def __init__(self, tabular: np.ndarray, targets: np.ndarray | None = None):
        self.tabular = np.asarray(tabular, dtype=np.float32)
        self.targets = None if targets is None else np.asarray(targets, dtype=np.float32)

    def __len__(self) -> int:
        return len(self.tabular)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        sample = {"tabular": torch.from_numpy(self.tabular[index])}
        if self.targets is not None:
            sample["target"] = torch.tensor(self.targets[index], dtype=torch.float32)
        return sample


## 3. Data Loading & Feature Engineering


In [ ]:
train_df = pd.read_csv(COMP_DATA_ROOT / 'train-metadata.csv')
test_df = pd.read_csv(COMP_DATA_ROOT / 'test-metadata.csv')
sample_sub = pd.read_csv(COMP_DATA_ROOT / 'sample_submission.csv')

print('Shape of train:', train_df.shape)
print('Shape of test:', test_df.shape)

# Apply feature engineering
print('Applying feature engineering...')
train_enhanced, test_enhanced = add_feature_engineering(
    train_df=train_df,
    test_df=test_df,
    group_col='patient_id',
    categorical_cols=DEFAULT_CATEGORICAL_COLS,
    use_contextual_group_norm_features=True,
    use_ugly_duckling_features=True,
    use_anatomical_zscore_features=True,
    use_area_aggregation_features=True,
)
print('Done. Shape after FE:', train_enhanced.shape)

# Filter columns available in both sets
numeric_cols, categorical_cols = select_available_columns(
    train_df=train_enhanced,
    test_df=test_enhanced,
    numeric_candidates=DEFAULT_NUMERIC_COLS,
    categorical_candidates=DEFAULT_CATEGORICAL_COLS,
)
print(f'Using {len(numeric_cols)} numeric and {len(categorical_cols)} categorical features.')


## 4. TabR Model Definition & Configurations
Dưới đây là cài đặt thuần PyTorch của TabR sử dụng `torch.cdist` thay vì thư viện FAISS nhị phân phức tạp, giúp notebook chạy độc lập, ổn định trên mọi môi trường và tự động tương thích với autograd.


In [ ]:
@dataclass
class TabRConfig:
    seed: int = 42
    n_splits: int = 5
    epochs: int = 15
    patience: int = 4
    lr: float = 3e-4
    weight_decay: float = 2e-4
    train_batch_size: int = 128
    valid_batch_size: int = 256
    negative_ratio: int = 6
    use_quantile_transform: bool = True
    quantile_noise_std: float = 1e-5
    
    # TabR specific params
    d_main: int = 96
    d_multiplier: float = 2.0
    context_size: int = 96
    context_dropout: float = 0.2
    dropout0: float = 0.1
    dropout1: float = 0.1

config = TabRConfig()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


In [ ]:
class TabRModel(nn.Module):
    def __init__(
        self,
        in_features: int,
        d_main: int = 96,
        d_multiplier: float = 2.0,
        encoder_n_blocks: int = 1,
        predictor_n_blocks: int = 1,
        context_dropout: float = 0.2,
        dropout0: float = 0.1,
        dropout1: float = 0.1,
        normalization: str = "LayerNorm",
        activation: str = "GELU",
    ):
        super().__init__()
        d_block = int(d_main * d_multiplier)
        Normalization = getattr(nn, normalization)
        Activation = getattr(nn, activation)

        def make_block(prenorm: bool) -> nn.Sequential:
            return nn.Sequential(
                *([Normalization(d_main)] if prenorm else []),
                nn.Linear(d_main, d_block),
                Activation(),
                nn.Dropout(dropout0),
                nn.Linear(d_block, d_main),
                nn.Dropout(dropout1),
            )

        self.linear = nn.Linear(in_features, d_main)
        self.blocks0 = nn.ModuleList([make_block(i > 0) for i in range(encoder_n_blocks)])
        self.normalization = Normalization(d_main)
        self.label_encoder = nn.Linear(1, d_main)
        self.K = nn.Linear(d_main, d_main)
        self.T = nn.Sequential(
            nn.Linear(d_main, d_block),
            Activation(),
            nn.Dropout(dropout0),
            nn.Linear(d_block, d_main, bias=False),
        )
        self.dropout = nn.Dropout(context_dropout)
        self.blocks1 = nn.ModuleList([make_block(True) for _ in range(predictor_n_blocks)])
        self.head = nn.Sequential(
            Normalization(d_main),
            Activation(),
            nn.Linear(d_main, 1),
        )

    def _encode(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.linear(x)
        for block in self.blocks0:
            x = x + block(x)
        k = self.K(self.normalization(x))
        return x, k

    def forward(
        self,
        x: torch.Tensor,
        y: Optional[torch.Tensor],
        candidate_x: torch.Tensor,
        candidate_y: torch.Tensor,
        context_size: int = 96,
        is_train: bool = True,
    ) -> torch.Tensor:
        candidate_x_encoded, candidate_k = self._encode(candidate_x)
        x_encoded, k = self._encode(x)

        batch_size = k.shape[0]
        device = k.device

        if is_train:
            assert y is not None
            # Append queries to the front of candidate keys
            candidate_k = torch.cat([k, candidate_k], dim=0)
            candidate_y = torch.cat([y, candidate_y], dim=0)

        # Fast pairwise L2 distance using torch.cdist
        dists = torch.cdist(k, candidate_k)

        if is_train:
            # Set self-distances to infinity to avoid target leakage during training
            dists[torch.arange(batch_size, device=device), torch.arange(batch_size, device=device)] = float('inf')

        # Find nearest neighbors
        distances, context_idx = torch.topk(dists, context_size, largest=False, dim=-1)

        context_k = candidate_k[context_idx]  # (B, context_size, D)
        context_y = candidate_y[context_idx]  # (B, context_size)

        # Compute similarity weights via softmax over negative L2 distances
        similarities = -torch.sum((k[:, None] - context_k) ** 2, dim=-1)
        probs = F.softmax(similarities, dim=-1)
        probs = self.dropout(probs)

        # Neighbors values projection
        context_y_emb = self.label_encoder(context_y[..., None])  # (B, context_size, D)
        diff = k[:, None] - context_k
        diff_proj = self.T(diff.flatten(0, 1)).view(batch_size, context_size, -1)
        values = context_y_emb + diff_proj

        # Attention aggregation: context_x = probs @ values
        context_x = torch.bmm(probs[:, None, :], values).squeeze(1)  # (B, D)

        # Residual fusion and prediction head
        x_fused = x_encoded + context_x
        for block in self.blocks1:
            x_fused = x_fused + block(x_fused)
        logits = self.head(x_fused).squeeze(1)
        return logits


In [ ]:
# Training and validation functions for TabR
def train_one_epoch_tabr(model, loader, candidate_x, candidate_y, optimizer, criterion, device, context_size):
    model.train()
    total_loss = 0.0
    seen = 0
    for batch in loader:
        tabular = batch["tabular"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        logits = model(
            x=tabular,
            y=targets,
            candidate_x=candidate_x,
            candidate_y=candidate_y,
            context_size=context_size,
            is_train=True
        )
        
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        
        batch_size = targets.size(0)
        total_loss += float(loss.item()) * batch_size
        seen += batch_size
    return total_loss / seen

@torch.inference_mode()
def evaluate_tabr(model, loader, candidate_x, candidate_y, criterion, device, context_size):
    model.eval()
    total_loss = 0.0
    seen = 0
    preds_all = []
    targets_all = []
    for batch in loader:
        tabular = batch["tabular"].to(device, non_blocking=True)
        targets = batch["target"].to(device, non_blocking=True)
        
        logits = model(
            x=tabular,
            y=None,
            candidate_x=candidate_x,
            candidate_y=candidate_y,
            context_size=context_size,
            is_train=False
        )
        
        loss = criterion(logits, targets)
        probs = torch.sigmoid(logits)
        
        preds_all.append(probs.cpu().numpy())
        targets_all.append(targets.cpu().numpy())
        
        batch_size = targets.size(0)
        total_loss += float(loss.item()) * batch_size
        seen += batch_size
        
    y_pred = np.concatenate(preds_all)
    y_true = np.concatenate(targets_all)
    try:
        auroc = float(roc_auc_score(y_true, y_pred))
    except ValueError:
        auroc = float("nan")
    pauc = partial_auc_score(y_true, y_pred)
    
    return {
        "loss": total_loss / seen,
        "auroc": auroc,
        "pauc": pauc,
        "preds": y_pred,
        "targets": y_true,
    }


## 5. Cross-Validation Training Loop


In [ ]:
set_seed(config.seed)
y = train_enhanced['target'].astype(np.int64).to_numpy()
groups = train_enhanced['patient_id'].to_numpy()

splitter = StratifiedGroupKFold(n_splits=config.n_splits, shuffle=True, random_state=config.seed)
folds = list(splitter.split(train_enhanced, y, groups))

oof_preds = np.zeros(len(train_enhanced), dtype=np.float32)
oof_mask = np.zeros(len(train_enhanced), dtype=bool)
test_preds = np.zeros(len(test_enhanced), dtype=np.float32)

for fold_idx, (train_idx, valid_idx) in enumerate(folds):
    print('=' * 80)
    print(f'Fold {fold_idx + 1}/{config.n_splits}')
    
    fold_train_df = train_enhanced.iloc[train_idx].reset_index(drop=True)
    fold_valid_df = train_enhanced.iloc[valid_idx].reset_index(drop=True)
    
    # Preprocess
    preprocessor = FoldTabularPreprocessor(
        numeric_cols=numeric_cols,
        categorical_cols=categorical_cols,
        use_quantile_transform=config.use_quantile_transform,
        quantile_noise_std=config.quantile_noise_std,
        quantile_random_state=config.seed + fold_idx + 1,
    )
    preprocessor.fit(fold_train_df)
    
    x_train = preprocessor.transform(fold_train_df)
    x_valid = preprocessor.transform(fold_valid_df)
    x_test = preprocessor.transform(test_enhanced)
    
    y_train = fold_train_df['target'].to_numpy(dtype=np.float32)
    y_valid = fold_valid_df['target'].to_numpy(dtype=np.float32)
    
    # For retrieval, we require train candidates as PyTorch tensors on GPU
    cand_x_tensor = torch.from_numpy(x_train).to(device)
    cand_y_tensor = torch.from_numpy(y_train).to(device)
    
    train_dataset = TabularDataset(x_train, y_train)
    valid_dataset = TabularDataset(x_valid, y_valid)
    test_dataset = TabularDataset(x_test)
    
    # Samplers for training (downsamples negatives)
    if 'tbp_lv_nevi_confidence' in fold_train_df.columns:
        nevi_conf = fold_train_df["tbp_lv_nevi_confidence"].fillna(50.0).to_numpy()
        hard_neg_weights = 0.1 + 0.9 * (((100.0 - nevi_conf) / 100.0) ** 2.0)
    else:
        hard_neg_weights = None
        
    sampler = make_balanced_sampler(
        targets=y_train,
        negative_ratio=config.negative_ratio,
        seed=config.seed + fold_idx,
        hard_negative_weights=hard_neg_weights,
    )
    
    train_loader = DataLoader(train_dataset, batch_size=config.train_batch_size, sampler=sampler, num_workers=0, pin_memory=True)
    valid_loader = DataLoader(valid_dataset, batch_size=config.valid_batch_size, shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=config.valid_batch_size, shuffle=False, num_workers=0, pin_memory=True)
    
    # Build model
    model = TabRModel(
        in_features=x_train.shape[1],
        d_main=config.d_main,
        d_multiplier=config.d_multiplier,
        context_dropout=config.context_dropout,
        dropout0=config.dropout0,
        dropout1=config.dropout1,
    ).to(device)
    
    n_pos = max(float(y_train.sum()), 1.0)
    n_neg = max(float(len(y_train) - y_train.sum()), 1.0)
    pos_weight = torch.tensor([np.sqrt(n_neg / n_pos)], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.lr, weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.epochs, eta_min=config.lr * 0.1)
    
    best_score = -np.inf
    best_state = None
    bad_epochs = 0
    
    for epoch in range(1, config.epochs + 1):
        train_loss = train_one_epoch_tabr(
            model=model,
            loader=train_loader,
            candidate_x=cand_x_tensor,
            candidate_y=cand_y_tensor,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            context_size=config.context_size,
        )
        val_metrics = evaluate_tabr(
            model=model,
            loader=valid_loader,
            candidate_x=cand_x_tensor,
            candidate_y=cand_y_tensor,
            criterion=criterion,
            device=device,
            context_size=config.context_size,
        )
        scheduler.step()
        
        print(f'Epoch {epoch:02d} | Train Loss: {train_loss:.5f} | Val Loss: {val_metrics["loss"]:.5f} | Val pAUC: {val_metrics["pauc"]:.5f}')
        
        if val_metrics["pauc"] > best_score:
            best_score = val_metrics["pauc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            
        if bad_epochs >= config.patience:
            print(f'Early stopping triggered.')
            break
            
    # Validation and test prediction on best state
    model.load_state_dict(best_state)
    val_metrics = evaluate_tabr(
        model=model,
        loader=valid_loader,
        candidate_x=cand_x_tensor,
        candidate_y=cand_y_tensor,
        criterion=criterion,
        device=device,
        context_size=config.context_size,
    )
    print(f'-> Best Fold Val pAUC: {val_metrics["pauc"]:.5f}')
    
    oof_preds[valid_idx] = val_metrics["preds"]
    oof_mask[valid_idx] = True
    
    # Inference on test set
    test_preds_fold = []
    with torch.no_grad():
        for batch in test_loader:
            tabular = batch["tabular"].to(device)
            logits = model(
                x=tabular,
                y=None,
                candidate_x=cand_x_tensor,
                candidate_y=cand_y_tensor,
                context_size=config.context_size,
                is_train=False
            )
            probs = torch.sigmoid(logits)
            test_preds_fold.append(probs.cpu().numpy())
    test_preds += np.concatenate(test_preds_fold) / config.n_splits
    
    del model, optimizer, scheduler, cand_x_tensor, cand_y_tensor
    torch.cuda.empty_cache()
    gc.collect()


## 6. Post-Processing & Submission Creation


In [ ]:
print('=' * 80)
cv_pauc = partial_auc_score(y[oof_mask], oof_preds[oof_mask])
print(f'OOF pAUC: {cv_pauc:.6f}')

# Write final submission file
submission = sample_sub.copy()
submission['target'] = test_preds
submission.to_csv('submission.csv', index=False)
print('Saved submission.csv successfully!')
display(submission.head())
